2


In [8]:


# Install required libraries if not present
import subprocess
import sys

try:
    import transformers
    import torch
    import accelerate
except ImportError:
    print("Installing required libraries: transformers, torch, accelerate...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers", "torch", "accelerate"])

from transformers import pipeline

# Load GPT-2 model
generator = pipeline(
    "text-generation",
    model="gpt2",
    device=-1  # Change to 0 if using GPU
)

# 1. Zero-shot prompt
zero_shot_prompt = (
    "Classify the sentiment of this review as Positive or Negative:\n"
    "Review: 'The product quality is excellent!'\n"
    "Sentiment:"
)

# 2. Few-shot prompt
few_shot_prompt = """
Review: 'I loved this movie, it was fantastic.'
Sentiment: Positive

Review: 'The service was slow and disappointing.'
Sentiment: Negative

Review: 'The product quality is excellent!'
Sentiment:
"""

# 3. Chain-of-Thought prompt
cot_prompt = """
Q: A shop had 15 apples. It sold 6 and then received 10 more. How many apples does it have now?
A: Let's think step by step.
15 - 6 = 9.
9 + 10 = 19.
The answer is 19.

Q: A library had 120 books. It lent out 45 and bought 30 new books. How many books does it have now?
A: Let's think step by step.
"""

# Store prompts in a list
prompts = [
    ("Zero-shot", zero_shot_prompt),
    ("Few-shot", few_shot_prompt),
    ("Chain-of-Thought", cot_prompt)
]

# Generate output for each prompt
for name, prompt in prompts:
    output = generator(
        prompt,
        max_new_tokens=40,
        num_return_sequences=1,
        do_sample=False
    )


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


6


In [9]:
# Install required libraries if not present
import subprocess
import sys

try:
    import faiss
except ImportError:
    print("Installing required libraries: faiss-cpu...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"])

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM # Changed from pipeline
# 1. Knowledge base
documents = [
    "The Eiffel Tower is located in Paris, France and was completed in 1889.",
    "Retrieval-Augmented Generation combines document retrieval with text generation.",
    "Python is a popular high-level programming language used in AI development.",
    "Vector databases store embeddings and support fast similarity search."
]
# 2. Embed documents
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = embed_model.encode(documents)
# 3. Build FAISS index
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings))
# 4. Query and retrieve top-2 relevant chunks
query = "What is RAG in AI?"
query_embedding = embed_model.encode([query])
D, I = index.search(np.array(query_embedding), k=2)
retrieved_chunks = [documents[i] for i in I[0]]
# 5. Build augmented prompt and generate answer
context = " ".join(retrieved_chunks)
prompt = f"Context: {context}\nQuestion: {query}\nAnswer:"

# --- MODIFIED PART: Using AutoTokenizer and AutoModelForSeq2SeqLM directly ---
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(
    inputs["input_ids"],
    max_new_tokens=60, # max_new_tokens controls the length of the generated text
    num_return_sequences=1
)
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
# --- END MODIFIED PART ---

print("Retrieved Context:", retrieved_chunks)
print("Answer:", answer) # Print the directly decoded answer

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Retrieved Context: ['Python is a popular high-level programming language used in AI development.', 'Retrieval-Augmented Generation combines document retrieval with text generation.']
Answer: combines document retrieval with text generation


7

In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
tokenizer = AutoTokenizer.from_pretrained("Salesforce/codegen-350M-mono")
model = AutoModelForCausalLM.from_pretrained("Salesforce/codegen-350M-mono")
def generate_code(prompt, max_new_tokens=80):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids
    output = model.generate(input_ids, max_new_tokens=max_new_tokens,
                            pad_token_id=tokenizer.eos_token_id, do_sample=False)
    return tokenizer.decode(output[0], skip_special_tokens=True)
#1.Code generation from a natural-language instruction
prompt1="#Write a Python function to check if a number is prime\ndef is_prime(n):"
print("Generated Function:\n", generate_code(prompt1))
#2.Debugging a faulty snippet
buggy_code="""#The following function should return the factorial of n, but has a bug. Fix it.
def factorial(n):
    result = 0
    for i in range(1, n+1):
        result = result * i
    return result
#Corrected function:
def factorial_fixed(n):"""
print("\nDebug Suggestion:\n", generate_code(buggy_code, max_new_tokens=60))

Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

[transformers] CodeGenForCausalLM LOAD REPORT from: Salesforce/codegen-350M-mono
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...19}.attn.causal_mask | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generated Function:
 #Write a Python function to check if a number is prime
def is_prime(n):
    if n == 2 or n == 3:
        return True
    if n % 2 == 0 or n % 3 == 0:
        return False
    for i in range(5, int(n**0.5)+1, 6):
        if n % i == 0:
            return False
    return True

#Write a Python function to

Debug Suggestion:
 #The following function should return the factorial of n, but has a bug. Fix it.
def factorial(n):
    result = 0
    for i in range(1, n+1):
        result = result * i
    return result
#Corrected function:
def factorial_fixed(n):
    result = 1
    for i in range(1, n+1):
        result = result * i
    return result

#The following function should return the factorial of n, but has a bug. Fix it.
def factorial_fixed2(n


8

In [11]:
from diffusers import StableDiffusionPipeline
import torch
try:
    pipe = StableDiffusionPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5"
    )
    pipe = pipe.to("cuda" if torch.cuda.is_available() else "cpu")
    prompt = "A futuristic city skyline at sunset, digital art, highly detailed"
    steps = 2 if not torch.cuda.is_available() else 30
    image = pipe(prompt, num_inference_steps=steps, guidance_scale=7.5).images[0]
    image.save("generated_city.png")
    print("Image generated and saved as generated_city.png")
except Exception as e:
    print(f"Skipping Stable Diffusion run: {str(e)}")
    from PIL import Image, ImageDraw
    img = Image.new('RGB', (512, 512), color=(73, 109, 137))
    d = ImageDraw.Draw(img)
    d.text((50, 240), "Stable Diffusion: Futuristic City Skyline", fill=(255, 255, 255))
    img.save("generated_city.png")
    print("Created fallback generated_city.png")

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

Potential NSFW content was detected in one or more images. A black image will be returned instead. Try again with a different prompt and/or seed.


Image generated and saved as generated_city.png


9

In [12]:
from transformers import BlipProcessor, BlipForConditionalGeneration, BlipForQuestionAnswering
from PIL import Image
import requests
image_url = "https://images.unsplash.com/photo-1519125323398-675f0ddb6308"
try:
    raw_image = Image.open(requests.get(image_url, stream=True).raw).convert("RGB")
except Exception:
    raw_image = Image.new("RGB", (300, 300), color=(100, 150, 200))
#---------- Image Captioning---------
cap_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
cap_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")
inputs = cap_processor(raw_image, return_tensors="pt")
caption_ids = cap_model.generate(**inputs, max_new_tokens=30)
caption = cap_processor.decode(caption_ids[0], skip_special_tokens=True)
print("Generated Caption:", caption)
#---------- Visual Question Answering---------
vqa_processor = BlipProcessor.from_pretrained("Salesforce/blip-vqa-base")
vqa_model = BlipForQuestionAnswering.from_pretrained("Salesforce/blip-vqa-base")
question = "What animal is in the picture?"
vqa_inputs = vqa_processor(raw_image, question, return_tensors="pt")
answer_ids = vqa_model.generate(**vqa_inputs)
answer = vqa_processor.decode(answer_ids[0], skip_special_tokens=True)
print("Question:", question)
print("Answer:", answer)

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Generated Caption: a toy chair sitting on a rock by the ocean


Loading weights:   0%|          | 0/788 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1625: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Question: What animal is in the picture?
Answer: bird
